### # 1(a) main_collect

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import json
import pymysql
import hashlib
import os
from dotenv import load_dotenv
from multiprocessing.dummy import Pool as ThreadPool

load_dotenv()
DB_PASSWORD = os.getenv('DB_PASSWORD', '')

CODES = ["005930", "000660", "035420", "051910", "005380", "006400", "035720", "068270", "105560", "055550"]
BATCH_SIZE = 500

START_DATE = datetime(2025, 1, 2)
END_DATE = datetime(2025, 12, 30)

def connect_db():
    return pymysql.connect(host='localhost', user='root', password=DB_PASSWORD, database='fsc_db', charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor)

def scrape_single_stock(code):
    url = "https://finance.naver.com/item/sise_day.naver"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    max_page = 100
    
    stock_rows = []
    is_finished = False
    
    for page in range(1, max_page + 1):
        if is_finished:
            break
            
        res = requests.get(url, headers=headers, params={"code": code, "page": page}, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")
        trs = soup.select("table.type2 tr")
        
        for tr in trs:
            tds = tr.select("td")
            if len(tds) < 7 or not tds[0].text.strip():
                continue
                
            date_str = tds[0].text.strip()
            row_date = datetime.strptime(date_str, "%Y.%m.%d")
            
            if row_date > END_DATE:
                continue
            if row_date < START_DATE:
                is_finished = True
                break
                
            diff_td = tds[2]
            img = diff_td.find('img')
            diff_val = diff_td.text.strip().replace(',', '')
            diff_text = f"{img['alt']} {diff_val}" if img and 'alt' in img.attrs else diff_val 
                
            stock_rows.append({
                'basDt': date_str,
                'srtnCd': code,
                'itmsNm': None, 
                'clpr': tds[1].text.strip(),
                'vs': diff_text,
                'mkp': tds[3].text.strip(),
                'hipr': tds[4].text.strip(),
                'lopr': tds[5].text.strip(),
                'trqu': tds[6].text.strip()
            })
        time.sleep(0.1)
        
    return stock_rows

def main_pipeline():
    start_time = time.time()
    all_data = []
    
    with ThreadPool(5) as pool:
        results = pool.map(scrape_single_stock, CODES)
        for rows in results:
            all_data.extend(rows)
            
    print(f"수집 완료: 총 {len(all_data)}건")

    now = datetime.now()
    source = 'naver_finance'
    url_src = "https://finance.naver.com/item/sise_day.naver"
    
    db_rows = []
    for item in all_data:
        payload = json.dumps(item, ensure_ascii=False)
        key_src = f"{source}|{item['basDt']}|{item['srtnCd']}"
        content_hash = hashlib.sha256(key_src.encode()).hexdigest()
        db_rows.append((source, url_src, now, payload, content_hash))

    if db_rows:
        conn = connect_db()
        try:
            with conn.cursor() as cur:
                cur.execute("SET SESSION unique_checks = 0;")
                cur.execute("SET SESSION foreign_key_checks = 0;")
                
                insert_sql = """
                    INSERT INTO raw_item(source, url, collected_at, payload, content_hash)
                    VALUES (%s, %s, %s, %s, %s)
                    ON DUPLICATE KEY UPDATE payload = VALUES(payload), collected_at = NOW()
                """
                
                for i in range(0, len(db_rows), BATCH_SIZE):
                    cur.executemany(insert_sql, db_rows[i:i + BATCH_SIZE])
                
                cur.execute("SET SESSION unique_checks = 1;")
                cur.execute("SET SESSION foreign_key_checks = 1;")
                
                conn.commit()
                print("Raw 테이블 적재 완료")
        except Exception as e:
            conn.rollback()
        finally:
            conn.close()

if __name__ == '__main__':
    main_pipeline()

수집 완료: 총 2420건
Raw 테이블 적재 완료


### # 1(b) main_cleansing

In [3]:
# 1(2)
import pymysql
import pandas as pd
import json
import re
import os
import sys
import logging
from datetime import datetime
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("cleansing_report.log", encoding="utf-8")
    ]
)
log = logging.getLogger("cleansing")

load_dotenv()
DB_PASSWORD = os.getenv('DB_PASSWORD', '')

SRC = {
    "nf": ("naver_finance", "tb_nf_stock"),
    "fsc": ("fsc_api", "tb_fsc_stock")
}
BATCH_SIZE = 500
COLS = ["bas_dt", "srtn_cd", "itms_nm", "clpr", "vs", "mkp", "hipr", "lopr", "trqu", "raw_id"]

def connect_db():
    return pymysql.connect(
        host='localhost', user='root', password=DB_PASSWORD, 
        database='fsc_db', charset='utf8mb4'
    )

def setup_tables(conn):
    create_queries = [
        """
        CREATE TABLE IF NOT EXISTS tb_nf_stock (
            bas_dt DATE NOT NULL, srtn_cd CHAR(6) NOT NULL, itms_nm VARCHAR(100), 
            clpr BIGINT, vs BIGINT, mkp BIGINT, hipr BIGINT, lopr BIGINT, trqu BIGINT, raw_id BIGINT,
            PRIMARY KEY (bas_dt, srtn_cd)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """,
        """
        CREATE TABLE IF NOT EXISTS tb_fsc_stock (
            bas_dt DATE NOT NULL, srtn_cd CHAR(6) NOT NULL, itms_nm VARCHAR(100), 
            clpr BIGINT, vs BIGINT, mkp BIGINT, hipr BIGINT, lopr BIGINT, trqu BIGINT, raw_id BIGINT,
            PRIMARY KEY (bas_dt, srtn_cd)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
    ]
    with conn.cursor() as cur:
        for q in create_queries: 
            cur.execute(q)
    conn.commit()

def to_num(t: str) -> int:
    m = re.search(r'\d+', re.sub(r'[^\d\-]', "", str(t)))
    return int(m.group()) if m else 0

def signed_vs(t: str) -> int:
    if pd.isna(t): return 0
    t_str = str(t)
    if '상승' in t_str: return to_num(t_str)
    elif '하락' in t_str or '-' in t_str: return -to_num(t_str)
    return to_num(t_str)

def to_date(d: str):
    d_str = str(d).replace('.', '')
    try:
        return datetime.strptime(d_str, '%Y%m%d')
    except ValueError:
        return None

def clean_naver(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "bas_dt": df["basDt"].apply(to_date),
        "srtn_cd": df["srtnCd"].astype(str).str.zfill(6),
        "itms_nm": None,
        "clpr": df["clpr"].apply(to_num),
        "vs": df["vs"].apply(signed_vs),
        "mkp": df["mkp"].apply(to_num),
        "hipr": df["hipr"].apply(to_num),
        "lopr": df["lopr"].apply(to_num),
        "trqu": df["trqu"].apply(to_num),
        "raw_id": df["raw_id"],
    })

def clean_fsc(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "bas_dt": df["basDt"].apply(to_date),
        "srtn_cd": df["srtnCd"].astype(str).str.zfill(6),
        "itms_nm": df["itmsNm"].astype(str).str.strip(),
        "clpr": df["clpr"].apply(to_num),
        "vs": df["vs"].apply(to_num),
        "mkp": df["mkp"].apply(to_num),
        "hipr": df["hipr"].apply(to_num),
        "lopr": df["lopr"].apply(to_num),
        "trqu": df["trqu"].apply(to_num),
        "raw_id": df["raw_id"],
    })

CLEANER = {"nf": clean_naver, "fsc": clean_fsc}

def run_cleansing(src_key: str):
    source, table = SRC[src_key]
    conn = connect_db()

    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM raw_item WHERE source = %s", (source,))
        total = cur.fetchone()[0]
    
    if not total:
        log.warning(f"[CLEAN:{src_key}] 데이터가 없습니다.")
        conn.close()
        return

    log.info(f"[CLEAN:{src_key}] Raw {total}건 → {table} (배치 {BATCH_SIZE})")

    insert_sql = f"""
    INSERT INTO {table}({','.join(COLS)}) 
        VALUES ({','.join(['%s']*len(COLS))})
        ON DUPLICATE KEY UPDATE clpr=VALUES(clpr), trqu=VALUES(trqu)
    """

    frames = []
    offset = 0
    stat = {"read": 0, "written": 0, "dup": 0}

    while offset < total:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT raw_id, payload FROM raw_item WHERE source = %s ORDER BY raw_id LIMIT %s OFFSET %s", 
                (source, BATCH_SIZE, offset)
            )
            chunk = cur.fetchall()

        if not chunk: break

        raw_df = pd.DataFrame([{**json.loads(p), "raw_id": rid} for rid, p in chunk])
        stat["read"] += len(raw_df)

        cleaned = CLEANER[src_key](raw_df)

        n0 = len(cleaned)
        cleaned = cleaned.drop_duplicates(subset=["bas_dt", "srtn_cd"], keep="last")
        stat["dup"] += n0 - len(cleaned)

        cleaned["bas_dt"] = cleaned["bas_dt"].dt.strftime("%Y-%m-%d")
        params = [tuple(row) for row in cleaned[COLS].values]
        
        with conn.cursor() as cur:
            cur.executemany(insert_sql, params)
        conn.commit()
        
        stat["written"] += len(params)
        frames.append(cleaned)
        offset += BATCH_SIZE
        
        log.info(f"  적재 {min(offset, total)} / {total}")

    report_summary(pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(), stat, src_key, table)
    conn.close()

def report_summary(df: pd.DataFrame, stat: dict, src: str, table: str):
    log.info(f"─────── 정제 요약 : {src} → {table} ───────")
    log.info(f"변환 건수      읽음 {stat['read']} · 적재 {stat['written']}")
    
    if df.empty:
        log.warning("적재된 행이 없습니다.")
        return

    d = pd.to_datetime(df["bas_dt"])
    log.info(f"기간           {d.min().date()} ~ {d.max().date()}")
    log.info(f"종목수         {df['srtn_cd'].nunique()}")

    dup_rate = stat["dup"] / max(stat["read"], 1) * 100
    log.info(f"중복률         {dup_rate:.2f}% (중복 제거 {stat['dup']}건)")

if __name__ == "__main__":
    conn = connect_db()
    setup_tables(conn)
    conn.close()
    
    for s in ['nf', 'fsc']:
        run_cleansing(s)

2026-08-28 15:21:12,242 [INFO] [CLEAN:nf] Raw 3225건 → tb_nf_stock (배치 500)
2026-08-28 15:21:13,063 [INFO]   적재 500 / 3225
2026-08-28 15:21:14,484 [INFO]   적재 1000 / 3225
2026-08-28 15:21:15,713 [INFO]   적재 1500 / 3225
2026-08-28 15:21:16,737 [INFO]   적재 2000 / 3225
2026-08-28 15:21:17,429 [INFO]   적재 2500 / 3225
2026-08-28 15:21:18,012 [INFO]   적재 3000 / 3225
2026-08-28 15:21:18,265 [INFO]   적재 3225 / 3225
2026-08-28 15:21:18,276 [INFO] ─────── 정제 요약 : nf → tb_nf_stock ───────
2026-08-28 15:21:18,279 [INFO] 변환 건수      읽음 3225 · 적재 3225
2026-08-28 15:21:18,308 [INFO] 기간           2025-01-02 ~ 2026-08-28
2026-08-28 15:21:18,313 [INFO] 종목수         10
2026-08-28 15:21:18,316 [INFO] 중복률         0.00% (중복 제거 0건)
2026-08-28 15:21:19,239 [INFO] [CLEAN:fsc] Raw 2420건 → tb_fsc_stock (배치 500)
2026-08-28 15:21:19,944 [INFO]   적재 500 / 2420
2026-08-28 15:21:20,595 [INFO]   적재 1000 / 2420
2026-08-28 15:21:21,149 [INFO]   적재 1500 / 2420
2026-08-28 15:21:21,683 [INFO]   적재 2000 / 2420
2026-08-28 15:21

### #2

In [ ]:
import pymysql
import pandas as pd
import os
import sys
import logging
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("mart_report.log", encoding="utf-8")
    ]
)
log = logging.getLogger("mart")

load_dotenv()
DB_PASSWORD = os.getenv('DB_PASSWORD', '')

DAILY_COLS = ["bas_dt", "srtn_cd", "clpr", "trqu", "chg_pct", "ma5", "ma20", "vol_ratio"]
MONTHLY_COLS = ["srtn_cd", "ym", "trd_days", "open_clpr", "close_clpr", "avg_clpr", "max_clpr", "min_clpr", "sum_trqu", "avg_trqu"]
BATCH_SIZE = 500

def connect_db():
    return pymysql.connect(
        host='localhost', user='root', password=DB_PASSWORD, 
        database='fsc_db', charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor
    )

def load_clean(table: str) -> pd.DataFrame:
    conn = connect_db()
    with conn.cursor() as cur:
        cur.execute(f"SELECT bas_dt, srtn_cd, clpr, trqu FROM {table}")
        rows = cur.fetchall()
    conn.close()

    if not rows:
        return pd.DataFrame()
    
    df = pd.DataFrame(rows)
    df['bas_dt'] = pd.to_datetime(df['bas_dt'])
    return df

def build_daily(df: pd.DataFrame) -> pd.DataFrame:
    d = df.sort_values(["srtn_cd", "bas_dt"]).copy()
    g = d.groupby("srtn_cd")

    d["chg_pct"] = (g["clpr"].pct_change(fill_method=None) * 100).round(2)
    d["ma5"] = g["clpr"].transform(lambda s: s.rolling(5).mean()).round(1)
    d["ma20"] = g["clpr"].transform(lambda s: s.rolling(20).mean()).round(1)

    vol_ma20 = g["trqu"].transform(lambda s: s.rolling(20).mean())
    d["vol_ratio"] = (d["trqu"] / vol_ma20).round(2)
    
    return d[DAILY_COLS]

def build_monthly(df: pd.DataFrame, min_days: int = 10) -> pd.DataFrame:
    d = df.sort_values(["srtn_cd", "bas_dt"]).copy()
    d["ym"] = d["bas_dt"].dt.strftime("%Y-%m")

    m = (d.groupby(["srtn_cd", "ym"], as_index=False)
           .agg(trd_days=("bas_dt", "count"),
                open_clpr=("clpr", "first"), 
                close_clpr=("clpr", "last"), 
                avg_clpr=("clpr", "mean"),
                max_clpr=("clpr", "max"),
                min_clpr=("clpr", "min"),
                sum_trqu=("trqu", "sum"),
                avg_trqu=("trqu", "mean")))

    before_cnt = len(m)
    m = m[m["trd_days"] >= min_days]
    
    if before_cnt != len(m):
        log.info(f" 거래일 {min_days}일 미만이므로 {before_cnt - len(m)}행 제외됨")

    for c in ("avg_clpr", "avg_trqu"):
        m[c] = m[c].round(0)
        
    return m[MONTHLY_COLS]

def insert_batch(table: str, cols: list, df: pd.DataFrame) -> int:
    if df.empty: return 0
    
    conn = connect_db()
    
    update_clause = f"{cols[-2]}=VALUES({cols[-2]})"
    sql = f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join(['%s'] * len(cols))}) ON DUPLICATE KEY UPDATE {update_clause}"

    out = df.copy()
    if "bas_dt" in out:
        out["bas_dt"] = pd.to_datetime(out["bas_dt"]).dt.strftime("%Y-%m-%d")

    params = [tuple(None if pd.isna(v) else v for v in row) for row in out[cols].values]
    
    try:
        for i in range(0, len(params), BATCH_SIZE):
            with conn.cursor() as cur:
                cur.executemany(sql, params[i:i + BATCH_SIZE])
            conn.commit()
    except Exception as e:
        log.error(f"오류 발생: {e}")
        conn.rollback()
    finally:
        conn.close()
        
    return len(params)

def main():
    table = 'tb_fsc_stock'
    df = load_clean(table)
    
    if df.empty:
        log.warning(f"{table}이 비어 있습니다.")
        return

    log.info(f"[MART] Clean {len(df)}행 ({df['bas_dt'].min().date()} ~ {df['bas_dt'].max().date()}, 종목 {df['srtn_cd'].nunique()})개")

    log.info(f"적재 완료: daily {insert_batch('tb_mart_stock_daily', DAILY_COLS, build_daily(df))}행, monthly {insert_batch('tb_mart_stock_monthly', MONTHLY_COLS, build_monthly(df))}행")
if __name__ == "__main__":
    main()

2026-08-28 15:35:52,461 [INFO] [MART] Clean 2420행 (2025-01-02 ~ 2025-12-30, 종목 10)개
2026-08-28 15:35:54,512 [INFO] 적재 완료: daily 2420행, monthly 120행
